# Create a general csv label table for clinical data

In [1]:
import pathlib
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from PIL import Image

## Load old mapping datasets

In [ ]:
# Clinical data
map_df_paths = list(pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\TUHH\Semester 6\Masterarbeit\OCT_clinical_data").glob('*.csv'))
map_df_paths = [p for p in map_df_paths if 'old' not in p.__str__()]
# [print(p) for p in map_df_paths]
cli_dfs = pd.concat([pd.read_csv(p) for p in map_df_paths], axis=0)
cli_dfs['img_relative_path'] = [pathlib.Path(p) for p in cli_dfs['img_relative_path']]
cli_dfs.loc[:, 'area'] = [p.parent.name.split('_')[-1] for p in cli_dfs['img_relative_path']]
# cli_dfs.loc[:, 'pat'] = [int(re.sub(r'[^\d]+', '', p.parts[1])) for p in cli_dfs['img_relative_path']]
cli_dfs.loc[:, 'trajectory'] = ['_'.join(p.stem.split('_')[:-3]) for p in cli_dfs.loc[:, 'img_relative_path']]
# cli_dfs
# cli_dfs.groupby(['subset', 'area', 'trajectory']).agg({'label': lambda x: ', '.join((set(x)))})

In [3]:
# train_areas = ["pat16", "pat13", "pat01", "pat15", "pat04", "pat09", "pat11", "pat12", "pat14", "pat06"]
train_areas = ['pat01', 'pat04', 'pat06', 'pat09', 'pat11', 'pat12', 'pat13', 'pat14', 'pat15', 'pat16']
valid_areas = ["pat02", "pat10"]
test_areas = ["pat08", "pat07"]
pat_split = {'train': train_areas,
                'valid': valid_areas,
                'test': test_areas}

In [ ]:
jpg_file_info = pd.read_csv('data_prep/clinical_jpg_files_info.csv')
jpg_file_info['trajectory'] = ['_'.join(t.split('_')[:-1]) for t in jpg_file_info['trajectory']]
jpg_file_info.groupby(['folder', 'trajectory']).agg({'label': lambda x: ', '.join((set(x)))})
old_labels = jpg_file_info.groupby(['folder', 'trajectory']).agg({'label': lambda x: ', '.join(set(x))}).reset_index()
# old_labels

In [5]:
histo_results = pd.read_excel('clinical_lesion_info_updated.xlsx')
histo_results = histo_results.drop(columns=[c for c in histo_results if re.search('Unnamed|link|.1|notes|side|location|laser', c)])
# Filter out rows to keep
# Remove files with no OCT data
histo_results = histo_results.loc[~((histo_results['filename'].isna()) | (histo_results['filename'] == '?')), :]
# Remove files where no lesion can be seen
histo_results = histo_results.loc[~(histo_results['malignity'] == 'healthy'), :]
# Get base trajecotry file name
histo_results['trajectory'] = ['_'.join(f.split('_')[:4]) for f in histo_results['filename']]
histo_results = histo_results.drop(columns=['filename'])
# Convert to old labeling scheme
histo_results['label'] = ['Healthy' if d is np.nan else 'Lesion' for d in histo_results['diagnosis']]
histo_results['diagnosis'] = histo_results['diagnosis'].fillna(histo_results['malignity'])
histo_results

,pat_id,malignity,diagnosis,trajectory,label
0,pat01,benign,benign,pat01_vc_re_run1,Healthy
1,pat02,malignant,SCC,pat02_vc_re_run1,Lesion
3,pat04,benign,benign,pat04_vc_re_run2,Healthy
4,pat04,benign,benign,pat04_vc_le_run1,Healthy
6,pat06,benign,benign,pat06_vc_re_run1,Healthy
7,pat06,benign,benign,pat06_vc_le_run1,Healthy
8,pat07,benign,benign,pat07_vc_le_run1,Healthy
9,pat08,malignant,"HSIL, SCC",pat08_VC_le_run4,Lesion
10,pat08,malignant,"HSIL, SCC",pat08_VC_le_run4,Lesion
12,pat08,malignant,HSIL,pat08_VC_re_run1,Lesion


In [6]:
histo_results.groupby('trajectory').agg({ 'malignity': lambda x: ', '.join(set(x)), 'label': lambda x: ', '.join(set(x))})

,malignity,label
trajectory,,
pat01_vc_re_run1,benign,Healthy
pat02_vc_re_run1,malignant,Lesion
pat04_vc_le_run1,benign,Healthy
pat04_vc_re_run2,benign,Healthy
pat06_vc_le_run1,benign,Healthy
pat06_vc_re_run1,benign,Healthy
pat07_vc_le_run1,benign,Healthy
pat08_VC_le_run4,malignant,Lesion
pat08_VC_re_run1,"malignant, benign","Lesion, Healthy"


In [7]:
lbl_comp = pd.merge(old_labels, histo_results, left_on=['folder', 'trajectory'], right_on=['pat_id', 'trajectory'], how='left', suffixes=['_old', '_new'])
lbl_comp[lbl_comp['label_old'] != lbl_comp['label_new']].dropna(subset=['label_new'])

,folder,trajectory,label_old,pat_id,malignity,diagnosis,label_new
0,pat01,pat01_vc_re_run1,Lesion,pat01,benign,benign,Healthy
5,pat04,pat04_vc_le_run1,Lesion,pat04,benign,benign,Healthy
7,pat04,pat04_vc_re_run2,Lesion,pat04,benign,benign,Healthy
9,pat06,pat06_vc_re_run1,Lesion,pat06,benign,benign,Healthy
10,pat07,pat07_vc_le_run1,Lesion,pat07,benign,benign,Healthy
18,pat08,pat08_VC_re_run1,Healthy,pat08,malignant,HSIL,Lesion
25,pat10,pat10_vc_re_run2,Lesion,pat10,benign,benign,Healthy
27,pat11,pat11_vc_le_run4,Lesion,pat11,benign,benign,Healthy
28,pat11,pat11_vc_le_run4,Lesion,pat11,benign,benign,Healthy


In [8]:
lbl_comp.dropna(subset=['label_new']).reset_index()

,index,folder,trajectory,label_old,pat_id,malignity,diagnosis,label_new
0,0,pat01,pat01_vc_re_run1,Lesion,pat01,benign,benign,Healthy
1,3,pat02,pat02_vc_re_run1,Lesion,pat02,malignant,SCC,Lesion
2,5,pat04,pat04_vc_le_run1,Lesion,pat04,benign,benign,Healthy
3,7,pat04,pat04_vc_re_run2,Lesion,pat04,benign,benign,Healthy
4,9,pat06,pat06_vc_re_run1,Lesion,pat06,benign,benign,Healthy
5,10,pat07,pat07_vc_le_run1,Lesion,pat07,benign,benign,Healthy
6,16,pat08,pat08_VC_le_run4,Lesion,pat08,malignant,"HSIL, SCC",Lesion
7,17,pat08,pat08_VC_le_run4,Lesion,pat08,malignant,"HSIL, SCC",Lesion
8,18,pat08,pat08_VC_re_run1,Healthy,pat08,malignant,HSIL,Lesion
9,19,pat08,pat08_VC_re_run1,Healthy,pat08,benign,benign,Healthy


In [9]:
cli_dfs

,img_relative_path,label,idx_start,idx_end,subset,area,trajectory
0,512mscans_sample20\pat07\pat07_vc_re_run2_1708...,Healthy,1,512,test,pat07,pat07_vc_re_run2
1,512mscans_sample20\pat07\pat07_vc_re_run2_1708...,Healthy,513,1024,test,pat07,pat07_vc_re_run2
2,512mscans_sample20\pat07\pat07_vc_re_run2_1708...,Healthy,1025,1536,test,pat07,pat07_vc_re_run2
3,512mscans_sample20\pat07\pat07_vc_re_run2_1708...,Healthy,1537,2048,test,pat07,pat07_vc_re_run2
4,512mscans_sample20\pat07\pat07_vc_re_run2_1708...,Healthy,2049,2560,test,pat07,pat07_vc_re_run2
...,...,...,...,...,...,...,...
298,512mscans_sample20\pat10\pat10_vc_le_run2_1721...,Healthy,31233,31744,valid,pat10,pat10_vc_le_run2
299,512mscans_sample20\pat10\pat10_vc_le_run2_1721...,Healthy,31745,32256,valid,pat10,pat10_vc_le_run2
300,512mscans_sample20\pat10\pat10_vc_le_run2_1721...,Healthy,32257,32768,valid,pat10,pat10_vc_le_run2
301,512mscans_sample20\pat10\pat10_vc_le_run2_1721...,Healthy,32769,33280,valid,pat10,pat10_vc_le_run2


In [10]:
jpg_file_info

,folder,path,img_relative_path,trajectory,label,idx_start,idx_end
0,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1,512
1,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,513,1024
2,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1025,1536
3,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1537,2048
4,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,2049,2560
...,...,...,...,...,...,...,...
2687,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,37377,37888
2688,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,37889,38400
2689,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,38401,38912
2690,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,38913,39424


In [129]:
jpg_file_info_new = pd.merge(jpg_file_info, histo_results, left_on=['folder', 'trajectory'], right_on=['pat_id', 'trajectory'], how='left', suffixes=['_old', '_new'])


In [130]:
jpg_file_info_new

,folder,path,img_relative_path,trajectory,label_old,idx_start,idx_end,pat_id,malignity,diagnosis,label_new
0,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1,512,pat01,benign,benign,Healthy
1,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,513,1024,pat01,benign,benign,Healthy
2,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1025,1536,pat01,benign,benign,Healthy
3,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1537,2048,pat01,benign,benign,Healthy
4,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,2049,2560,pat01,benign,benign,Healthy
...,...,...,...,...,...,...,...,...,...,...,...
2928,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,37377,37888,pat16,malignant,SCC,Lesion
2929,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,37889,38400,pat16,malignant,SCC,Lesion
2930,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,38401,38912,pat16,malignant,SCC,Lesion
2931,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,38913,39424,pat16,malignant,SCC,Lesion


In [134]:
jpg_file_info_new[~jpg_file_info_new['diagnosis'].isna()]['folder'].unique()

<StringArray>
['pat01', 'pat02', 'pat04', 'pat06', 'pat07', 'pat08', 'pat09', 'pat10',
 'pat11', 'pat13', 'pat15', 'pat16']
Length: 12, dtype: str

In [131]:
jpg_file_info_new.groupby('diagnosis').agg({'folder': 'nunique', 'path': 'count'})

,folder,path
diagnosis,,
HSIL,2,238
"HSIL, SCC",1,192
SCC,3,247
benign,9,846


In [136]:
jpg_file_info_new

,folder,path,img_relative_path,trajectory,label_old,idx_start,idx_end,pat_id,malignity,diagnosis,label_new
0,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1,512,pat01,benign,benign,Healthy
1,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,513,1024,pat01,benign,benign,Healthy
2,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1025,1536,pat01,benign,benign,Healthy
3,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,1537,2048,pat01,benign,benign,Healthy
4,pat01,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat01/pat01_vc_re_run1_1702...,pat01_vc_re_run1,Lesion,2049,2560,pat01,benign,benign,Healthy
...,...,...,...,...,...,...,...,...,...,...,...
2928,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,37377,37888,pat16,malignant,SCC,Lesion
2929,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,37889,38400,pat16,malignant,SCC,Lesion
2930,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,38401,38912,pat16,malignant,SCC,Lesion
2931,pat16,/data/Boudreault/OCT_clinical_data/512mscans_s...,512mscans_sample20/pat16/pat16_vc_re_run4_1769...,pat16_vc_re_run4,Lesion,38913,39424,pat16,malignant,SCC,Lesion


In [137]:
jpg_file_info_new.dropna(subset=['label_new']).groupby('folder').agg({'diagnosis': lambda x: ', '.join(set(x))})

,diagnosis
folder,
pat01,benign
pat02,SCC
pat04,benign
pat06,benign
pat07,benign
pat08,"benign, HSIL, SCC, HSIL"
pat09,benign
pat10,benign
pat11,benign


In [143]:
histo_results.sort_values(['pat_id', 'trajectory', 'label'], ascending=True).drop_duplicates(subset=['trajectory'], keep='last')

,pat_id,malignity,diagnosis,trajectory,label
0,pat01,benign,benign,pat01_vc_re_run1,Healthy
1,pat02,malignant,SCC,pat02_vc_re_run1,Lesion
4,pat04,benign,benign,pat04_vc_le_run1,Healthy
3,pat04,benign,benign,pat04_vc_re_run2,Healthy
7,pat06,benign,benign,pat06_vc_le_run1,Healthy
6,pat06,benign,benign,pat06_vc_re_run1,Healthy
8,pat07,benign,benign,pat07_vc_le_run1,Healthy
10,pat08,malignant,"HSIL, SCC",pat08_VC_le_run4,Lesion
12,pat08,malignant,HSIL,pat08_VC_re_run1,Lesion
16,pat09,benign,benign,pat09_vc_le_run1,Healthy


In [139]:
histo_results[histo_results['pat_id'] == 'pat08']

,pat_id,malignity,diagnosis,trajectory,label
9,pat08,malignant,"HSIL, SCC",pat08_VC_le_run4,Lesion
10,pat08,malignant,"HSIL, SCC",pat08_VC_le_run4,Lesion
12,pat08,malignant,HSIL,pat08_VC_re_run1,Lesion
13,pat08,benign,benign,pat08_VC_re_run1,Healthy


In [86]:
pat_split

{'train': ['pat01',
  'pat04',
  'pat06',
  'pat09',
  'pat11',
  'pat12',
  'pat13',
  'pat14',
  'pat15',
  'pat16'],
 'valid': ['pat02', 'pat10'],
 'test': ['pat08', 'pat07']}

In [84]:
old_labels.groupby('label').agg({'folder': lambda x: ', '.join(set(x))}).to_dict()

{'folder': {'Healthy': 'pat07, pat16, pat02, pat11, pat04, pat10, pat15, pat09, pat08, pat01',
  'Lesion': 'pat07, pat13, pat15, pat16, pat02, pat06, pat04, pat10, pat11, pat08, pat01'}}

In [85]:
lbl_comp.groupby('label_new').agg({'folder': lambda x: ', '.join(set(x))}).to_dict()

{'folder': {'Healthy': 'pat07, pat16, pat06, pat04, pat10, pat11, pat09, pat08, pat01',
  'Lesion': 'pat13, pat16, pat02, pat15, pat08'}}

In [153]:
1/1e-3

1000.0

## Check matching Endoscope-OCT recordings

In [41]:
oct_root_path = pathlib.Path(r"\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\patient_study\OCTData")
oct_files = list(oct_root_path.rglob('*.mat'))
oct_files = pd.DataFrame(oct_files, columns=['full_path'])
oct_files['pat_id'] = [p.parent.stem for p in oct_files['full_path']]
oct_files['run_name'] = ['_'.join(p.stem.split('_')[:3]) for p in oct_files['full_path']]
oct_files['run_id'] = ['' if 'run' not in p.stem else [r for r in p.stem.split('_') if 'run' in r][0] for p in oct_files['full_path']]
oct_files['Name OCT'] = [f"{n}_{i}" if i != '' else f"{n}" for n, i in  zip(oct_files['run_name'], oct_files['run_id'])]
oct_files['cropped'] = ['crop' in p.stem for p in oct_files['full_path']]


In [43]:
endo_root_path = pathlib.Path(r"\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\patient_study\EndoskopData")
endo_files = list(endo_root_path.rglob('*.mp4'))
endo_files = pd.DataFrame(endo_files, columns=['full_path'])
endo_files['pat_id'] = [p.parent.stem for p in endo_files['full_path']]
endo_files['run_name'] = ['_'.join(p.stem.split('_')[:3]) for p in endo_files['full_path']]
endo_files['run_id'] = ['' if 'run' not in p.stem else [r for r in p.stem.split('_') if 'run' in r][0] for p in endo_files['full_path']]
endo_files['Name Video'] = [f"{n}_{i}" if i != '' else f"{n}" for n, i in  zip(endo_files['run_name'], endo_files['run_id'])]
endo_files = endo_files[endo_files['pat_id'] != 'eval'].copy()

In [19]:
oct_files[(oct_files['pat_id'] == 'pat08') & (oct_files['cropped'] == False)]['full_path'].tolist()

[WindowsPath('//elwing.et8.tuhh.de/mtec-share/Latus/UKE_HNO/patient_study/OCTData/pat08/pat08_ary_re_run1_1718959836696494_raw.mat'),
 WindowsPath('//elwing.et8.tuhh.de/mtec-share/Latus/UKE_HNO/patient_study/OCTData/pat08/pat08_VC_le_run1_1718959452389813_raw.mat'),
 WindowsPath('//elwing.et8.tuhh.de/mtec-share/Latus/UKE_HNO/patient_study/OCTData/pat08/pat08_VC_le_run2_1718959514122534_raw.mat'),
 WindowsPath('//elwing.et8.tuhh.de/mtec-share/Latus/UKE_HNO/patient_study/OCTData/pat08/pat08_VC_le_run3_1718959785114081_raw.mat'),
 WindowsPath('//elwing.et8.tuhh.de/mtec-share/Latus/UKE_HNO/patient_study/OCTData/pat08/pat08_VC_le_run4_1718960174746080_raw.mat'),
 WindowsPath('//elwing.et8.tuhh.de/mtec-share/Latus/UKE_HNO/patient_study/OCTData/pat08/pat08_VC_re_run1_1718960120367610_raw.mat')]

## Check endoscope video quality

In [4]:
from datetime import datetime, timedelta
start = datetime(2026,7,13,8,30)
delay = timedelta(hours=7, minutes=48)
# delay = timedelta(hours=10, minutes=0)
delay2 = timedelta(minutes=30)
end = start + delay + delay2
print(end)

2026-07-13 16:48:00


In [83]:
rgba_img = pathlib.Path(r"C:\Users\Ana Jade\OneDrive\Documents\Ecole\RTG3144\Projects\TMI\OCT_RGB_mapping\data\rgba\pat01\pat01_vc_re_run1\21.png")
rgba_img = Image.open(rgba_img)
gray = rgba_img.convert('LA')
rgba_img.show()
gray.show()


In [69]:
rgba_img.shape

(1080, 1920, 4)

In [71]:
rgba_img.dtype

dtype('uint8')

In [64]:
gray

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1080, 1920), dtype=uint8)

In [ ]:
endo_vids = 

0         pat01_allRegions
1                     n.V.
2    pat01_vc_re_NO_lesion
3         pat01_allRegions
4         pat01_allRegions
             ...          
7        pat18_epi_ce_run2
0       pat19_epi_mid_run1
1         pat19_vc_le_run1
2         pat19_vc_le_run2
3         pat19_vc_le_run4
Name: Name Video, Length: 101, dtype: str

In [ ]:
video_path = pathlib.Path("\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\patient_study\EndoskopData\pat19\pat19_vc_le_run1_framegrabber.mp4")


,full_path,pat_id,Name OCT,cropped
0,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat01,pat01_ary_le,False
1,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat01,pat01_ary_re,False
2,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat01,pat01_vall_re,True
3,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat01,pat01_vall_re,True
4,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat01,pat01_vall_re,True
...,...,...,...,...
236,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat18,pat18_vc_re,False
237,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat19,pat19_epi_mid,False
238,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat19,pat19_vc_le,False
239,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat19,pat19_vc_le,False


In [33]:
oct_files[oct_files['pat_id'] == 'pat03']

,full_path,pat_id,Name OCT,run,cropped
65,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat03,pat03_bot_re,run1,False
66,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat03,pat03_vc_re,run1,False
67,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat03,pat03_vc_re,run2,False
68,\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\...,pat03,pat03_vc_re,run3,False


## Load labels excel file

In [28]:
full_lbl_path = pathlib.Path(r"\\elwing.et8.tuhh.de\mtec-share\Latus\UKE_HNO\patient_study\Protokoll_UKE_HNO_Patienten_Pathologie_inklZeiten.xlsx")
full_lbl_file = pd.ExcelFile(full_lbl_path)
full_lbl_sheets = full_lbl_file.sheet_names
print(full_lbl_sheets)

['Übersicht', 'Aufnahmezeiten', 'Vorlage', 'pat01_20231211', 'pat02_20243101', 'pat03_20240206', 'pat04_20240207', 'pat05_20240215', 'pat06_20240226', 'pat07_20240226', 'pat08_20240621', 'pat09_20240625', 'pat10_20240725', 'pat11_20240823', 'pat12_20250110', 'pat13_20250125', 'pat14_20250408', 'pat15_20251201', 'pat16_20260123', 'pat17_20260330', 'pat18_20260410', 'pat19_20260420']


In [29]:
all_pat_df = []
for pat_sheet in full_lbl_sheets[3:]:
    pat_id = pat_sheet.split('_')[0]
    pat_df = pd.read_excel(full_lbl_path, sheet_name=pat_sheet, header=2)
    # Remove extra rows
    # pat_df = pat_df.dropna(subset=['Name OCT', 'Name Video'], how='all')
    pat_df = pat_df.dropna(axis=0, thresh=4) # Drop any extra rows that don't have at least 4 non-nan values
    # print(f"{pat_id}: {pat_df.columns}")
    # Check if runs have been renamed
    if 'renamed' in pat_df.columns:
        # Overwrite Name OCT with renamed column
        pat_df.loc[~pat_df['renamed'].isna(), 'Name OCT'] = pat_df.loc[~pat_df['renamed'].isna(), 'renamed']
        pat_df = pat_df.drop(columns=['renamed'])
    # Add patient id 
    pat_df = pat_df.drop(columns=['Unnamed: 0'])
    pat_df.insert(0, 'pat_id', pat_id)
    all_pat_df.append(pat_df)
all_pat_df = pd.concat(all_pat_df, axis=0)
all_pat_df = all_pat_df.dropna(axis=1, how='all')
all_pat_df = all_pat_df.astype({'Klinische Beurteilung': str, 
                                'Diagnose (OCT)': str,
                                'Diagnose (Histo)': str,
                                'Kommentar': str})
all_pat_df

,pat_id,Region,Name OCT,Name Video,Startzeit,Einstellbarkeit OCT,Klinische Beurteilung,OCT Bildqualität,Diagnose (Weißlicht),Diagnose (OCT),Diagnose (Histo),Kommentar,Datum
0,pat01,VC rechts,pat01_vc_re_run1,pat01_allRegions,09:00:00,4.0,Läsion,3,NaN,aufällig,negativer Befund,"Läsion deutlich erkennbar, kurzer Datensatz",2023-12-11
1,pat01,VC links,n.V.,n.V.,09:01:00,6.0,keine,keine,NaN,NaN,NaN,"schwierig einzustellen vom winkel, keine Bildd...",NaT
2,pat01,Unveränderter Bereich VC rechts,pat01_vc_re_run2,pat01_vc_re_NO_lesion,09:01:00,4.0,keine,3,NaN,unaufällig,NaN,sehr kurz,NaT
3,pat01,Ary rechts,pat01_ary_re_run1,pat01_allRegions,09:02:00,2.0,keine,2,NaN,unaufällig,NaN,NaN,NaT
4,pat01,Ary links,pat01_ary_le_run1,pat01_allRegions,09:03:00,2.0,keine,2,NaN,unaufällig,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,pat18,epi_ce,pat18_epi_ce_run2,pat18_epi_ce_run2,08:30:00,3.0,ohne Befund,3.0,NaN,NaN,NaN,NaN,NaT
0,pat19,epi mid,pat19_epi_mid_run1,pat19_epi_mid_run1,08:26:00,3.0,ohne Befund,3.0,NaN,NaN,NaN,NaN,NaT
1,pat19,vc le,pat19_vc_le_run1,pat19_vc_le_run1,08:32:00,5.0,verdächtig,3.0,NaN,NaN,NaN,"Alle VC runs nur sehr kurz, evtl. nicht Zielre...",NaT
2,pat19,vc le,pat19_vc_le_run2,pat19_vc_le_run2,08:34:00,6.0,verdächtig,3.0,NaN,NaN,NaN,NaN,NaT


In [ ]:
area_codes = {'VC': 'Vocal Cords',
              'BOT': 'Bottom of Tongue',
              'Ary': 'Ary Cartilage',
              'Vall': 'Vallecula',
              'Epi': 'Epiglottis',
}
lesion_info = {''}

In [5]:
all_pat_df[all_pat_df['pat_id'] == 'pat02']

,pat_id,Region,Name OCT,Name Video,Startzeit,Einstellbarkeit OCT,Klinische Beurteilung,OCT Bildqualität,Diagnose (Weißlicht),Diagnose (OCT),Diagnose (Histo),Kommentar,Datum
0,pat02,Lä VC rechts,pat02_vc_re_run1,pat02_vc_re_run1_cut,11:46:00,2.0,Läsion,2.0,NaN,aufällig,positiver Befund,NaN,NaT
1,pat02,unverändert VC links,pat02_vc_le_run1,pat02_vc_le_run1,NaN,1.5,keine,2.0,NaN,unaufällig,NaN,NaN,NaT
2,pat02,Lä VC rechts,pat02_vc_re_run2,pat02_vc_re_run2,NaN,1.0,Läsion,1.0,NaN,aufällig,positiver Befund,NaN,NaT
3,pat02,Vall rechts,pat02_vall_re_run1,pat02_vall_re_run1,NaN,2.0,keine,1.0,NaN,unaufällig,NaN,NaN,NaT
4,pat02,Vall links,pat02_vall_le_run1,pat02_vall_le_run1,11:48:00,2.0,keine,2.5,NaN,unaufällig,NaN,NaN,NaT


In [56]:
all_pat_df[all_pat_df['Diagnose (OCT)'].str.contains('aufällig')]

,pat_id,Region,Name OCT,Name Video,Startzeit,Einstellbarkeit OCT,Klinische Beurteilung,OCT Bildqualität,Diagnose (Weißlicht),Diagnose (OCT),Diagnose (Histo),Kommentar,Datum
0,pat01,VC rechts,pat01_vc_re_run1,pat01_allRegions,09:00:00,4.0,Läsion,3,NaN,aufällig,negativer Befund,"Läsion deutlich erkennbar, kurzer Datensatz",2023-12-11
2,pat01,Unveränderter Bereich VC rechts,pat01_vc_re_run2,pat01_vc_re_NO_lesion,09:01:00,4.0,keine,3,NaN,unaufällig,NaN,sehr kurz,NaT
3,pat01,Ary rechts,pat01_ary_re_run1,pat01_allRegions,09:02:00,2.0,keine,2,NaN,unaufällig,NaN,NaN,NaT
4,pat01,Ary links,pat01_ary_le_run1,pat01_allRegions,09:03:00,2.0,keine,2,NaN,unaufällig,NaN,NaN,NaT
5,pat01,Vall rechts,pat01_vall_le_run1,pat01_allRegions,09:04:00,1.0,keine,1,NaN,unaufällig,NaN,NaN,NaT
0,pat02,Lä VC rechts,pat02_vc_re_run1,pat02_vc_re_run1_cut,11:46:00,2.0,Läsion,2.0,NaN,aufällig,positiver Befund,NaN,NaT
1,pat02,unverändert VC links,pat02_vc_le_run1,pat02_vc_le_run1,NaN,1.5,keine,2.0,NaN,unaufällig,NaN,NaN,NaT
2,pat02,Lä VC rechts,pat02_vc_re_run2,pat02_vc_re_run2,NaN,1.0,Läsion,1.0,NaN,aufällig,positiver Befund,NaN,NaT
3,pat02,Vall rechts,pat02_vall_re_run1,pat02_vall_re_run1,NaN,2.0,keine,1.0,NaN,unaufällig,NaN,NaN,NaT
4,pat02,Vall links,pat02_vall_le_run1,pat02_vall_le_run1,11:48:00,2.0,keine,2.5,NaN,unaufällig,NaN,NaN,NaT


In [8]:
all_pat_df.groupby('Region').agg({'pat_id': lambda x: ', '.join((set(x))), 'Name OCT': 'count'})

,pat_id,Name OCT
Region,,
Ary,pat08,1
Ary li,pat10,1
Ary links,"pat01, pat04",2
Ary re,pat10,1
Ary re + le,pat07,1
Ary rechts,"pat01, pat04",2
BOT rechts,pat03,1
Lä VC rechts,pat02,2
Unveränderter Bereich VC rechts,pat01,1


In [78]:
all_pat_df.groupby('Klinische Beurteilung').agg({'pat_id': lambda x: ', '.join((set(x))), 'Name OCT': 'count'})

,pat_id,Name OCT
Klinische Beurteilung,,
Leukoplakie,pat10,2
Läsion,"pat02, pat03, pat11, pat01",8
Läsion Gefäß,pat04,4
Läsion vorhanden,pat05,2
Rinke Ödem (Wasseranlagerung),pat09,1
Zyste,pat01,1
auffälliger Bereich,pat03,1
keine,"pat02, pat01",7
keine,pat01,1


In [74]:
all_pat_df.groupby('Diagnose (OCT)').agg({'pat_id': lambda x: ', '.join((set(x))), 'Name OCT': 'count'})

,pat_id,Name OCT
Diagnose (OCT),,
aufällig,"pat02, pat01",3
unaufällig,"pat02, pat01",7


In [73]:
all_pat_df.groupby('Diagnose (Histo)').agg({'pat_id': lambda x: ', '.join((set(x))), 'Name OCT': 'count'})

,pat_id,Name OCT
Diagnose (Histo),,
negativer Befund,pat01,1
ohne Befund,pat04,4
positiver Befund,pat02,2


In [7]:
all_pat_df[(all_pat_df['Klinische Beurteilung'] == 'Läsion') # & 
           # (all_pat_df['Diagnose (OCT)'].str.contains('aufällig')) & 
           # (all_pat_df['Diagnose (Histo)'].str.contains('negativ', case=False))
           ]

,pat_id,Region,Name OCT,Name Video,Startzeit,Einstellbarkeit OCT,Klinische Beurteilung,OCT Bildqualität,Diagnose (Weißlicht),Diagnose (OCT),Diagnose (Histo),Kommentar,Datum
0,pat01,VC rechts,pat01_vc_re_run1,pat01_allRegions,09:00:00,4.0,Läsion,3,NaN,aufällig,negativer Befund,"Läsion deutlich erkennbar, kurzer Datensatz",2023-12-11
0,pat02,Lä VC rechts,pat02_vc_re_run1,pat02_vc_re_run1_cut,11:46:00,2.0,Läsion,2.0,NaN,aufällig,positiver Befund,NaN,NaT
2,pat02,Lä VC rechts,pat02_vc_re_run2,pat02_vc_re_run2,NaN,1.0,Läsion,1.0,NaN,aufällig,positiver Befund,NaN,NaT
0,pat03,VC rechts,pat03_vc_re_run1,pat03_vc_re_run1,NaN,5.0,Läsion,6.0,NaN,NaN,NaN,"sehr viele Bewegungen Patientin, daher Einstel...",NaT
1,pat03,VC rechts,pat03_vc_re_run2,pat03_vc_re_run2,NaN,6.0,Läsion,5.0,NaN,NaN,NaN,sehr kurzer Datensatz,NaT
2,pat03,VC rechts,pat03_vc_re_run3,pat03_vc_re_run3,NaN,6.0,Läsion,6.0,NaN,NaN,NaN,NaN,NaT
0,pat11,vc li,pat11_vc_le_run2,pat11_vc_li_run2,10:58:00,1.0,Läsion,1,NaN,NaN,NaN,"run1 fehlt, da Server abgestürzt",NaT
6,pat11,vc li,pat11_vc_le_run4,pat11_vc_li_run4_move,11:06:00,3.0,Läsion,4,NaN,NaN,NaN,versucht über Läsion zu bewegen; run3 fehlt da...,NaT


In [61]:
all_pat_df[(all_pat_df['Klinische Beurteilung'] == 'Läsion') & 
           # (all_pat_df['Diagnose (OCT)'].str.contains('aufällig')) & 
           (all_pat_df['Diagnose (Histo)'].str.contains('positiv', case=False))]

,pat_id,Region,Name OCT,Name Video,Startzeit,Einstellbarkeit OCT,Klinische Beurteilung,OCT Bildqualität,Diagnose (Weißlicht),Diagnose (OCT),Diagnose (Histo),Kommentar,Datum
0,pat02,Lä VC rechts,pat02_vc_re_run1,pat02_vc_re_run1_cut,11:46:00,2.0,Läsion,2.0,NaN,aufällig,positiver Befund,NaN,NaT
2,pat02,Lä VC rechts,pat02_vc_re_run2,pat02_vc_re_run2,NaN,1.0,Läsion,1.0,NaN,aufällig,positiver Befund,NaN,NaT


In [12]:
area_codes = {'VC': 'Vocal Cords',
              'BOT': 'Bottom of Tongue',
              'Ary': 'Ary Cartilage',
              'Vall': 'Vallecula',
              'Epi': 'Epiglottis',
}

In [28]:
pat_sheet = full_lbl_sheets[5]
############################
# in for loop
############################
pat_id = pat_sheet.split('_')[0]
print(f"pat id: {pat_id}")
pat_df = pd.read_excel(full_lbl_path, sheet_name=pat_sheet, header=2)
# Remove extra rows
pat_df = pat_df.dropna(subset=['Name OCT', 'Name Video'], how='all')
# Check if runs have been renamed
if 'renamed' in pat_df.columns:
    # Overwrite Name OCT with renamed column
    pat_df.loc[~pat_df['renamed'].isna(), 'Name OCT'] = pat_df.loc[~pat_df['renamed'].isna(), 'renamed']
    pat_df = pat_df.drop(columns=['renamed'])
# Add patient id 
pat_df = pat_df.drop(columns=['Unnamed: 0'])
pat_df.insert(0, 'pat_id', pat_id)
pat_df

pat id: pat03


,pat_id,Region,Name OCT,Name Video,Startzeit,Einstellbarkeit OCT,Klinische Beurteilung,OCT Bildqualität,Diagnose (Weißlicht),Diagnose (OCT),Diagnose (Histo),Kommentar
0,pat03,VC rechts,pat03_vc_re_run1,pat03_vc_re_run1,NaN,5.0,Läsion,6.0,NaN,NaN,NaN,"sehr viele Bewegungen Patientin, daher Einstel..."
1,pat03,VC rechts,pat03_vc_re_run2,pat03_vc_re_run2,NaN,6.0,Läsion,5.0,NaN,NaN,NaN,sehr kurzer Datensatz
2,pat03,VC rechts,pat03_vc_re_run3,pat03_vc_re_run3,NaN,6.0,Läsion,6.0,NaN,NaN,NaN,NaN
3,pat03,BOT rechts,pat03_bot_re_run1,pat03_vc_re_run1,NaN,4.0,auffälliger Bereich,5.0,NaN,NaN,NaN,NaN


In [27]:
pat_df.columns

Index(['Unnamed: 0', 'Region', 'Name OCT', 'Name Video', 'Startzeit',
       'Einstellbarkeit OCT', 'Klinische Beurteilung', 'OCT Bildqualität',
       'Diagnose (Weißlicht)', 'Diagnose (OCT)', 'Diagnose (Histo)',
       'Kommentar'],
      dtype='str')